# Additional Models
## TODO:
- Try Decision Tree and Random Forest regressors. 
- Compare their test R² against baseline. 
- Document model behavior (strengths/weaknesses). 

# Decision Tree Regressors
Implementing the CART algorithm, with help from https://www.geeksforgeeks.org/machine-learning/python-decision-tree-regression-using-sklearn/ 

Reducing overfitting with help from https://ameersaleem.substack.com/p/what-overfitting-looks-like-in-decision 

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor, export_graphviz
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn import tree

from IPython.display import Image

In [16]:
# Importing Data
root = "C:/Users/donutii/Desktop/IDX-Exchange-Internship-Data-Science-61"
data_location = f"{root}/IDX_Exchange/deliverables"

testing_set = pd.read_csv(f'{data_location}/CRMLS_202605_testing_set.csv')
training_set = pd.read_csv(f'{data_location}/CRMLS_202505_202604_training_set.csv') 

# Splitting it 
    # Testing
vars_testing = testing_set.drop(columns='ClosePrice')
target_testing = testing_set['ClosePrice']

In [17]:
# Splitting training set into different months to test hyperparameter
months = ['2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']
training_set_months = []
for month in range(len(months)):
    training_set_months.append(training_set[training_set['CloseDate'].str.contains(months[month])])
    #(training_set_months[month].columns)
    training_set_months[month].drop(columns='CloseDate')
    
    
    # X
vars = training_set.drop(columns=['ClosePrice', 'CloseDate'])
target = training_set['ClosePrice']


In [18]:
# Beginning Decision Tree Regression
    # (Note: absolute_error was tanking performance, so I removed it). I'll test it later.

# Pre-pruning
criterion = ['squared_error', 'friedman_mse', 'poisson']
depths = [20, 30, 40, 60]
#min_samples_leaves = [0.1, 0.5, 0.99] # did not improve r2 value at all

model_criterias = []
model_depth = []
model_training_r2s = []
model_testing_rq = []


for criteria in criterion:
    # Testing different max depths as well
    for depth in depths:
        
        model = DecisionTreeRegressor(max_depth=depth, criterion=criteria, random_state=42).fit(vars, target)

        # print results
        r_sq = model.score(vars, target)
        r_sq2 = model.score(vars_testing, target_testing)
        
        model_criterias.append(criteria)
        model_depth.append(model.get_depth())
        
        model_training_r2s.append(r_sq)
        model_testing_rq.append(r_sq2)
                
            
decision_tree_results = pd.DataFrame(data={'Tree Criteria': model_criterias, 'Model Depth': model_depth, 
                                           'Training R2': model_training_r2s, 'Testing R2': model_testing_rq})
print(decision_tree_results)

    Tree Criteria  Model Depth  Training R2  Testing R2
0   squared_error           20     0.984527    0.839397
1   squared_error           30     0.999564    0.833977
2   squared_error           40     0.999997    0.835598
3   squared_error           46     0.999999    0.832324
4    friedman_mse           20     0.984528    0.842166
5    friedman_mse           30     0.999564    0.833154
6    friedman_mse           40     0.999997    0.826933
7    friedman_mse           46     0.999999    0.833447
8         poisson           20     0.987045    0.837741
9         poisson           30     0.999748    0.827185
10        poisson           40     0.999999    0.827850
11        poisson           48     0.999999    0.827895


Conclusion:

Generally, the model tended to greatly overfit. However, the efficacy of Poisson was far better, almost having an R2 score of 0.5. They definitely performed better than the baseline model though,

# Random Forest Regressor
Made with the help of https://www.geeksforgeeks.org/machine-learning/random-forest-regression-in-python/ 


In [19]:
# Importing
from sklearn.ensemble import RandomForestRegressor

In [20]:
model = RandomForestRegressor(criterion='poisson', max_features='log2', random_state=42).fit(vars, target)

# print results
r_sq = model.score(vars, target)
r_sq2 = model.score(vars_testing, target_testing)

print(f'Training R2: {r_sq}')
print(f'Testing R2: {r_sq2}')

Training R2: 0.986813885041428
Testing R2: 0.9015859656575079


Conclusion:

This model had a very long runtime (around 4 minutes), which made it somewhat difficult to tinker with the hyperparameters. It also performed worse than the decision tree on both accounts, which isn't ideal. It also seems to perform better